# Pinecone Local — Connect Examples

[Pinecone Local](https://docs.pinecone.io/guides/operations/local-development) is an in-memory Pinecone emulator that runs in Docker. It is great for development and CI because:

- No Pinecone account or real API key is required (use any placeholder key).
- Data lives in memory only — it is **wiped when the container stops**.
- The same `pinecone` Python client works against it; you just point `host` at localhost.

This notebook shows two connection styles:

1. **Index-per-container** — start one `pinecone-local` container per index with a fixed host/port.
2. **Controller/dynamic** — run the emulator and create indexes at runtime, discovering their host.

> Ports used below: `5080` for the controller, `5081+` for individual indexes.

## 0. Start Pinecone Local

Run **one** of the options below in a terminal before executing the notebook cells.

### Option A — Docker run (single index)
```bash
docker run -d --name pinecone-local \
  -e PORT=5080 \
  -e PINECONE_HOST=http://localhost \
  -p 5080-5090:5080-5090 \
  --platform linux/amd64 \
  ghcr.io/pinecone-io/pinecone-local:latest
```

### Option B — docker-compose
```yaml
services:
  pinecone-local:
    image: ghcr.io/pinecone-io/pinecone-local:latest
    platform: linux/amd64
    environment:
      PORT: 5080
      PINECONE_HOST: http://localhost
    ports:
      - "5080-5090:5080-5090"
```
```bash
docker compose up -d pinecone-local
```

## 1. Install the client

The `[grpc]` extra is optional but gives faster upserts/queries. Pinecone Local supports both HTTP and gRPC.

In [ ]:
%pip install -q "pinecone[grpc]>=5.4.0"

## 2. Connect to the controller

Against Pinecone Local you pass a placeholder `api_key` and set `host` to the local controller. `ssl=False` (or an `http://` host) tells the client not to use TLS.

In [11]:
import os
from pinecone import Pinecone, ServerlessSpec

# Any non-empty string works as the key for the local emulator.
API_KEY = os.getenv("PINECONE_API_KEY", "pclocal")
CONTROLLER_HOST = os.getenv("PINECONE_CONTROLLER_HOST", "http://localhost:5081")

pc = Pinecone(api_key=API_KEY, host=CONTROLLER_HOST)

# Sanity check: listing indexes should not raise.
print("Existing indexes:", [ix["name"] for ix in pc.list_indexes()])

Existing indexes: []


## 3. Create an index

Pinecone Local only supports **serverless** indexes. The `cloud`/`region` values are ignored by the emulator but still required by the API.

In [12]:
INDEX_NAME = "agentmesh-demo"
DIMENSION = 8  # small toy dimension for the examples

if not pc.has_index(INDEX_NAME):
    pc.create_index(
        name=INDEX_NAME,
        dimension=DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1"),
    )

desc = pc.describe_index(INDEX_NAME)


def local_host(raw: str) -> str:
    """Pinecone Local reports hosts like '0.0.0.0:5082' with no scheme.
    Rewrite to a client-connectable http URL on localhost."""
    host = raw.replace("0.0.0.0", "localhost")
    if not host.startswith(("http://", "https://")):
        host = "http://" + host
    return host


INDEX_HOST = local_host(desc["host"])
print("Raw host:   ", desc["host"])      # e.g. 0.0.0.0:5082
print("Index host: ", INDEX_HOST)         # e.g. http://localhost:5082
print("Status:     ", desc["status"])


Raw host:    0.0.0.0:5082
Index host:  http://localhost:5082
Status:      {'ready': True, 'state': 'Ready'}


In [5]:
desc

{
    "name": "agentmesh-demo",
    "metric": "cosine",
    "host": "0.0.0.0:5082",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 8,
    "deletion_protection": "disabled"
}

## 4. Get an index handle and upsert vectors

Use the host returned by `describe_index` so the client talks directly to the index's port.

In [13]:
index = pc.Index(host=INDEX_HOST)

index.upsert(
    vectors=[
        {"id": "doc-1", "values": [0.1] * DIMENSION, "metadata": {"topic": "agents", "lang": "en"}},
        {"id": "doc-2", "values": [0.2] * DIMENSION, "metadata": {"topic": "search", "lang": "en"}},
        {"id": "doc-3", "values": [0.3] * DIMENSION, "metadata": {"topic": "agents", "lang": "fr"}},
    ],
    namespace="examples",
)

print(index.describe_index_stats())


{'dimension': 8,
 'metric': 'cosine',
 'namespaces': {'examples': {'vector_count': 3}},
 'total_vector_count': 3,
 'vector_type': 'dense'}


## 5. Query — nearest neighbours

In [14]:
res = index.query(
    vector=[0.15] * DIMENSION,
    top_k=3,
    namespace="examples",
    include_metadata=True,
)
for match in res["matches"]:
    print(f"{match['id']:>6}  score={match['score']:.4f}  {match.get('metadata')}")

 doc-1  score=1.0000  {'lang': 'en', 'topic': 'agents'}
 doc-2  score=1.0000  {'lang': 'en', 'topic': 'search'}
 doc-3  score=1.0000  {'lang': 'fr', 'topic': 'agents'}


## 6. Query with a metadata filter

Only return English documents about agents.

In [15]:
res = index.query(
    vector=[0.15] * DIMENSION,
    top_k=3,
    namespace="examples",
    include_metadata=True,
    filter={"topic": {"$eq": "agents"}, "lang": {"$eq": "en"}},
)
for match in res["matches"]:
    print(f"{match['id']:>6}  score={match['score']:.4f}  {match.get('metadata')}")

 doc-1  score=1.0000  {'lang': 'en', 'topic': 'agents'}


## 7. Example 2 — gRPC client

Same emulator, faster transport. Useful for bulk upserts. The API is identical to the HTTP client.

In [16]:
from pinecone.grpc import PineconeGRPC

pc_grpc = PineconeGRPC(api_key=API_KEY, host=CONTROLLER_HOST)
index_grpc = pc_grpc.Index(host=INDEX_HOST)

# Batch upsert via gRPC
batch = [(f"bulk-{i}", [i / 100] * DIMENSION, {"batch": True}) for i in range(50)]
index_grpc.upsert(vectors=batch, namespace="bulk")
print(index_grpc.describe_index_stats())


PineconeException: UNAVAILABLE:errors resolving http://localhost:5082: [field:hostname lookup error:address lookup failed for http://localhost:5082: Misformatted domain name]

## 8. Cleanup

Delete vectors and the index. (Stopping the container also discards everything.)

In [ ]:
index.delete(delete_all=True, namespace="examples")
index.delete(delete_all=True, namespace="bulk")

if pc.has_index(INDEX_NAME):
    pc.delete_index(INDEX_NAME)
print("Cleaned up.")